In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

def identify_case_control_county(target_pathogen):
    # === 1. 读取原始数据 ===
    data = pd.read_excel("../1_clean/data_merge_all_20260109.xlsx")
    data = data[~(data['host_species'].isna() & data['standard virus name'].isna())].reset_index(drop=True) #key

    # 1. 先获取所有 gb
    all_gb = data[['gb']].drop_duplicates()

    # 2. 生成 case/control 表（只针对有病毒记录的行）
    case_control_df = (
        data.assign(presence=1)
        .pivot_table(
            index="gb",
            columns="standard virus name",
            values="presence",
            aggfunc="max",
            fill_value=0
        )
        .reset_index()
    )

    # 3. 合并，确保所有 gb 都在里面
    case_control_df = (
        all_gb
        .merge(case_control_df, on="gb", how="left")
        .fillna(0)
    )

    # print(target_pathogen, case_control_df[case_control_df[target_pathogen] == 1].shape[0], case_control_df[case_control_df[target_pathogen] == 0].shape[0])
    case_control_df["status"] = case_control_df[target_pathogen]

    # === 3. major reservoir ===
    if target_pathogen == 'Seoul virus':
        major_reservoir = ['Rattus norvegicus']
    elif target_pathogen == 'Hantaan virus':
        major_reservoir = ['Apodemus agrarius']
    elif target_pathogen == 'Wenzhou arenavirus':
        major_reservoir = ['Rattus norvegicus', 'Suncus murinus']
    elif target_pathogen == 'Rat hepatitis E virus':
        major_reservoir = ['Rattus']
    elif target_pathogen == 'Encephalomyocarditis virus':
            major_reservoir = ['Rattus']
    elif target_pathogen == 'Betacoronavirus':
            major_reservoir = ['Apodemus agrarius']           
    elif target_pathogen == 'Morbillivirus':
            major_reservoir = ['Apodemus agrarius']    
    elif target_pathogen == 'Henipavirus':
            major_reservoir = ['Crocidura', 'Suncus murinus'] 

    # === 4. 阳性县区 ===
    case_county_list = case_control_df[case_control_df['status'] == 1]['gb'].tolist()

    # === 5. 潜在阴性县区 ===
    potential_control_county_list = case_control_df[case_control_df['status'] == 0]['gb'].tolist()

    # print(len(np.unique(data['gb'])), len(potential_control_county_list))

    # === 6. 预先读取所有 host 高风险县，避免重复读文件 ===
    host_hrc_dict = {}
    for host in major_reservoir:
        hrc_file = f'output/4_predict/host/prob/{host}.xlsx'
        if Path(hrc_file).exists():
            hrc_gb_df = pd.read_excel(hrc_file)
            hrc_gb_df = hrc_gb_df[hrc_gb_df['pred'] >= hrc_gb_df["threshold"]]
            host_hrc_dict[host] = set(hrc_gb_df['gb'])
        else:
            print(f"⚠ 高风险县文件不存在: {hrc_file}")
            host_hrc_dict[host] = set()

    # === 7. 筛选真正的阴性县（不在 host 高风险县） ===
    control_county_list = []
    for gb_code in potential_control_county_list:
        if not any(gb_code in host_hrc_dict[host] for host in major_reservoir):
            control_county_list.append(gb_code)

    # === 8. 保存结果 ===
    df_case = pd.DataFrame({'gb': case_county_list, f'{pathogen}': [1]*len(case_county_list)})
    df_control = pd.DataFrame({'gb': control_county_list, f'{pathogen}': [0]*len(control_county_list)})
    df_all = pd.concat([df_case, df_control], ignore_index=True)

    output_dir = Path('output/4_predict/pathogen/case_control')
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / f'case_control_{target_pathogen}.xlsx'
    df_all.to_excel(output_file, index=False)
    print(f"病原体：{pathogen}, 阳性县区：{df_case.shape[0]}, 阴性县区：{df_control.shape[0]}")


# === 批量处理符合条件的病原体 ===
targets = ['Hantaan virus', 'Seoul virus', 'Wenzhou arenavirus', 'Encephalomyocarditis virus', 'Rat hepatitis E virus',  'Betacoronavirus', 'Morbillivirus', 'Henipavirus']

for pathogen in targets:
    identify_case_control_county(pathogen)

病原体：Hantaan virus, 阳性县区：71, 阴性县区：778
病原体：Seoul virus, 阳性县区：183, 阴性县区：358
病原体：Wenzhou arenavirus, 阳性县区：20, 阴性县区：255
病原体：Encephalomyocarditis virus, 阳性县区：14, 阴性县区：312
病原体：Rat hepatitis E virus, 阳性县区：36, 阴性县区：312
病原体：Betacoronavirus, 阳性县区：58, 阴性县区：750
病原体：Morbillivirus, 阳性县区：26, 阴性县区：780
病原体：Henipavirus, 阳性县区：24, 阴性县区：677


In [9]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from pathlib import Path
import os

# ============================================================
#                       单次 Grid Search
# ============================================================
def run_grid_search(target_pathogen, risk_factor_df, weight_df, case_control_df):
    # 风险因子列
    all_risk_cols = risk_factor_df.columns.drop("gb")

    # 合并数据
    all_data = (
        risk_factor_df
        .merge(weight_df, on="gb", how="left")
        .merge(
            case_control_df[["gb", target_pathogen]].rename(columns={target_pathogen: "status"}),
            on="gb",
            how="left"
        )
    )

    survey_data = all_data.dropna(subset=["status"]).reset_index(drop=True)

    # train-test split（仅训练集，保持固定随机种子）
    train, _ = train_test_split(
        survey_data,
        test_size=0.25,
        random_state=42,
        stratify=survey_data["status"],
        shuffle=True
    )

    X_train = train[all_risk_cols].values
    y_train = train["status"].values
    w_train = train["rescaled"].values

    # XGBoost 模型
    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        tree_method="hist",
        n_jobs=1
    )

    # 参数网格
    param_grid = {
        "n_estimators": [3000],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.001, 0.005, 0.01],
        "subsample": [0.7, 0.75, 0.8],
    }

    grid = GridSearchCV(
        model,
        param_grid,
        scoring="neg_log_loss",
        cv=5,
        n_jobs=max(1, int(os.cpu_count() * 0.7))
    )

    grid.fit(X_train, y_train, sample_weight=w_train)

    result = {"species": target_pathogen, **grid.best_params_}
    return result

# ============================================================
#                       主程序
# ============================================================
output_dir = Path('output/3_grid_search')
output_dir.mkdir(parents=True, exist_ok=True)

# 权重
weight_df = pd.read_excel("output/2_weight/weight.xlsx")[["gb", "rescaled"]]

# 风险因子 baseline
risk_factor_df = pd.read_excel("output/1_risk_factor/risk_factor.xlsx")

# 读取目标 pathogen
targets = ['Hantaan virus', 'Seoul virus', 'Wenzhou arenavirus', 'Encephalomyocarditis virus', 'Rat hepatitis E virus',  'Betacoronavirus', 'Morbillivirus', 'Henipavirus']


results_list = []

for idx, target_pathogen in enumerate(targets, 1):
    case_control_file = f'output/4_predict/pathogen/case_control/case_control_{target_pathogen}.xlsx'
    case_control_df = pd.read_excel(case_control_file)

    record = run_grid_search(
        target_pathogen,
        risk_factor_df,
        weight_df,
        case_control_df
    )
    results_list.append(record)
    print(f"✓ [{idx}/{len(targets)}] {target_pathogen:30s} done.")

# 保存结果
results_df = pd.DataFrame(results_list)
results_file = output_dir / 'pathogen_best_params.xlsx'
results_df.to_excel(results_file, index=False)
print(f"\n=== 完成：所有 pathogen 物种 baseline 情景已写入 {results_file} ===")

✓ [1/8] Hantaan virus                  done.
✓ [2/8] Seoul virus                    done.
✓ [3/8] Wenzhou arenavirus             done.
✓ [4/8] Encephalomyocarditis virus     done.
✓ [5/8] Rat hepatitis E virus          done.
✓ [6/8] Betacoronavirus                done.
✓ [7/8] Morbillivirus                  done.
✓ [8/8] Henipavirus                    done.

=== 完成：所有 pathogen 物种 baseline 情景已写入 output\3_grid_search\pathogen_best_params.xlsx ===
